# Step 05 — Fine-tuning (QLoRA) Qwen2.5-3B-Instruct

Train QLoRA adapters on a single CUDA GPU. A 16 GB GPU may require a smaller context budget; 4-bit weights do not bound activation memory.

This notebook installs pinned training libraries, loads optional Colab Secrets, reserves the held-out test set, reports excluded duplicates/conflicting annotations, selects a stratified subset, and checks every chat-formatted example before training. Loss is computed only on the target JSON.

In Colab, checkpoints are saved to Google Drive by default. Each experiment uses a fresh base model, a unique output directory, and a unique Hub revision. Set an existing `RUN_ID` and `RESUME_FROM_CHECKPOINT=True` to resume that experiment.

W&B is optional. `HF_TOKEN` is required only when `PUSH_TO_HUB=True`. `TYPESAFE_API_KEY` is needed for full semantic evaluation; without it, inspect schema validity and programmatic metrics and semantic-error statuses.

After dependency installation, restart the kernel/runtime if an already-imported library changed. See `ISSUES.md` in this folder for fixes, remaining constraints, and validation.

In [ ]:
# Colab runtime availability is controlled by Colab.
# Durable checkpoints, rather than browser keep-alive scripts, provide recovery.

In [ ]:
# Sync the Colab checkout; preserve an existing local checkout.
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Muhammad-Zain01/resume-parser-model.git'
CLONE_DIR = Path('/content/resume-parser-model')
MARKER = 'src/steps/05_fine_tuning/fine_tuning.py'
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if (CLONE_DIR / '.git').is_dir():
        subprocess.run(['git', '-C', str(CLONE_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)], check=True)
    PROJECT_ROOT = CLONE_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / MARKER).is_file():
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            raise FileNotFoundError('Open this notebook from inside the project.')
        PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
# Pin the training APIs. Restart the runtime if a loaded package changes.
%pip install -q -r {PROJECT_ROOT}/requirements.txt

In [ ]:
import json
import os
import sys
from importlib import import_module, reload
from importlib.metadata import version

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=False)

# Avoid cached project code when re-running after a source update.
reload(import_module('src.steps.05_fine_tuning.fine_tuning'))
fine_tuning = reload(import_module('src.steps.05_fine_tuning'))
report_results = import_module('src.steps.04_evaluation').report_results
runtime_versions = {}
for package in ['torch', 'transformers', 'trl', 'peft', 'accelerate', 'datasets', 'bitsandbytes', 'wandb']:
    installed = version(package)
    runtime_versions[package] = installed
    loaded = getattr(sys.modules.get(package), '__version__', installed)
    if str(loaded) != installed:
        raise RuntimeError(f'{package} changed after import. Restart the runtime and rerun the notebook.')
    print(f'{package}: {installed}')
print('Fine-tuning module:', fine_tuning.__file__)

### Secrets -> environment variables

Pulls the project keys from **Google Colab's Secrets manager** (the key icon in the left sidebar) and exposes them as environment variables for every later cell. Colab values take precedence over `.env`. Outside Colab this is a no-op, so the `.env` file is used instead.

In [ ]:
# Keys this step reads from Colab Secrets (or .env). Repo ids live in the code,
# not here. Extras are skipped harmlessly.
SECRET_KEYS = [
    'HF_TOKEN',             # download gated assets + push the adapter
    'HF_DATASET_REVISION',  # optional: dataset revision (defaults to main)
    'WANDB_API_KEY',        # Weights & Biases login
    'WANDB_ENTITY',         # optional: W&B entity (has a default)
    'WANDB_PROJECT',        # optional: W&B project (has a default)
    'TYPESAFE_API_KEY',     # step-04 semantic judge during evaluation
]


def load_colab_secrets(keys=SECRET_KEYS):
    try:
        from google.colab import userdata  # available only inside Colab
    except ImportError:
        print('Not running in Colab; using .env / existing environment instead.')
        return {}
    loaded = {}
    for key in keys:
        try:
            value = userdata.get(key)
        except Exception:
            continue  # not defined or notebook access not granted
        if value:
            os.environ[key] = value  # Colab Secrets take precedence over .env
            loaded[key] = value
    return loaded


colab_secrets = load_colab_secrets()
print('Secrets loaded from Colab:', ', '.join(sorted(colab_secrets)) or 'none')

In [ ]:
# Reporting and publishing can be disabled without requiring credentials.
USE_WANDB = True
PUSH_TO_HUB = False
RUN_BASELINE = False  # Set True for a directly comparable before/after evaluation.

required_keys = []
if USE_WANDB:
    required_keys.append('WANDB_API_KEY')
if PUSH_TO_HUB:
    required_keys.append('HF_TOKEN')
for key in SECRET_KEYS:
    print(f'{key}:', 'set' if os.getenv(key) else 'unset')
missing = [key for key in required_keys if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing required keys: {', '.join(missing)}. Add them to Colab Secrets or .env.")
if not os.getenv('TYPESAFE_API_KEY'):
    print('Semantic evaluation is unavailable. Inspect programmatic metrics and semantic-error statuses.')

In [ ]:
MODEL_ID = fine_tuning.DEFAULT_MODEL_ID
LOAD_IN_4BIT = True
EVAL_SAMPLE = 20
MAX_NEW_TOKENS = 3072
WANDB_ENTITY = os.getenv('WANDB_ENTITY')  # Use the account's default entity if unset.
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'resume-parser')

print('HF model repo:', fine_tuning.model_repo_id())
print('HF dataset repo:', fine_tuning.dataset_repo_id())

### Hyperparameters and checkpoint storage

For an independent experiment, rerun the hyperparameter, model-loading, dataset/configuration, and training cells. The model-loading cell deletes the previous model/trainer references and reloads base weights.

For recovery after a runtime reset, set `RUN_ID` to the previous printed run ID and `RESUME_FROM_CHECKPOINT=True` (latest checkpoint), or a checkpoint path. Keep the dataset revision, selection seed, and hyperparameters unchanged. A checkpoint contains optimizer/scheduler/RNG state; the final adapter export alone is not a resumable checkpoint.

Colab mounts Google Drive when `PERSIST_CHECKPOINTS=True`. To use another durable location, set `FINETUNING_OUTPUT_ROOT`. Turning persistence off leaves checkpoints in ephemeral runtime storage.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

HYPERPARAMETERS = {
    'learning_rate': 2e-4,
    'epochs': 2,
    'per_device_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 16,
    'max_seq_length': 8192,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'warmup_ratio': 0.03,
    'lr_scheduler_type': 'cosine',
    'save_steps': 100,
    'eval_steps': 100,
    'max_train_samples': 2000,  # Stratified selection after cleaning and length checks.
    'seed': 42,
}
OVERLENGTH_POLICY = 'drop'  # Exclusions are reported; no target JSON is truncated.
PERSIST_CHECKPOINTS = True
RUN_TAG = 'lr2e4-ep2-r16'
RUN_ID = None  # To resume, paste the previous run ID here.
RESUME_FROM_CHECKPOINT = None  # True = latest checkpoint; or use an explicit path.

if RESUME_FROM_CHECKPOINT and not RUN_ID:
    raise ValueError('Set the original RUN_ID before resuming.')
EXPERIMENT_ID = RUN_ID or f"{RUN_TAG}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex[:8]}"
RUN_NAME = f'qwen2.5-3b-resume-parser-qlora-{EXPERIMENT_ID}'
root_override = os.getenv('FINETUNING_OUTPUT_ROOT')
if root_override:
    output_root = Path(root_override).expanduser()
elif IN_COLAB and PERSIST_CHECKPOINTS:
    from google.colab import drive
    drive.mount('/content/drive')
    output_root = Path('/content/drive/MyDrive/resume-parser-model/models')
else:
    output_root = fine_tuning.MODELS_DIR
    if IN_COLAB:
        print('Checkpoints are ephemeral: a runtime reset can delete them.')
OUTPUT_DIR = output_root / RUN_NAME
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not RESUME_FROM_CHECKPOINT:
    raise ValueError('Output directory already contains a run. Choose a new RUN_ID or resume explicitly.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Run ID (save for recovery):', EXPERIMENT_ID)
print('Output dir:', OUTPUT_DIR)
print(json.dumps(HYPERPARAMETERS, indent=2))

## 1. Download, validate, and prepare the splits

The test split is downloaded only to reserve its resume identities/text. It supplies no training targets. Training excludes all held-out overlap; duplicate texts are deduplicated, and groups with conflicting targets are excluded with counts. Original dataset files are preserved.

In [ ]:
splits = fine_tuning.download_curated_splits(splits=('train', 'validation', 'test'))
raw_train_records = fine_tuning.load_records(splits['train'])
raw_validation_records = fine_tuning.load_records(splits['validation'])
test_records = fine_tuning.load_records(splits['test'])
train_records, validation_records, preparation_report = fine_tuning.prepare_training_records(
    raw_train_records, raw_validation_records, test_records=test_records,
)
print(json.dumps(preparation_report, indent=2))

## 2. Load fresh base weights and check complete examples

QLoRA requires CUDA. Precision follows GPU capability (FP16 on T4, BF16 on supported GPUs), and the model stays on one device. Every prepared training/validation record is checked with the actual chat template before subset selection. Overlength examples are explicitly excluded or rejected according to `OVERLENGTH_POLICY`.

In [ ]:
# Delete live references before collecting memory between experiments.
for name in ('trainer', 'model', 'tokenizer'):
    globals().pop(name, None)
fine_tuning.free_memory()
model, tokenizer = fine_tuning.load_model_and_tokenizer(MODEL_ID, load_in_4bit=LOAD_IN_4BIT)
context_limit = getattr(model.config, 'max_position_embeddings', None)
if context_limit and HYPERPARAMETERS['max_seq_length'] > context_limit:
    raise ValueError('max_seq_length exceeds the model context window.')
fit_train_records, train_lengths = fine_tuning.filter_records_by_length(
    train_records, tokenizer, max_length=HYPERPARAMETERS['max_seq_length'], policy=OVERLENGTH_POLICY,
)
fit_validation_records, validation_lengths = fine_tuning.filter_records_by_length(
    validation_records, tokenizer, max_length=HYPERPARAMETERS['max_seq_length'], policy=OVERLENGTH_POLICY,
)
print('Train length checks:', json.dumps(train_lengths, indent=2))
print('Validation length checks:', json.dumps(validation_lengths, indent=2))

## 3. LoRA and training configuration

Build conversational prompt/completion datasets with loss only on the assistant JSON. Evaluation batch size is explicitly 1. The 2,000-record subset is sampled proportionally across categories, including each category when the sample budget permits.

In [ ]:
import hashlib
lora_config = fine_tuning.build_lora_config(
    r=HYPERPARAMETERS['lora_r'],
    alpha=HYPERPARAMETERS['lora_alpha'],
    dropout=HYPERPARAMETERS['lora_dropout'],
)
training_args = fine_tuning.build_training_arguments(
    OUTPUT_DIR,
    epochs=HYPERPARAMETERS['epochs'],
    learning_rate=HYPERPARAMETERS['learning_rate'],
    per_device_batch_size=HYPERPARAMETERS['per_device_batch_size'],
    per_device_eval_batch_size=HYPERPARAMETERS['per_device_eval_batch_size'],
    gradient_accumulation_steps=HYPERPARAMETERS['gradient_accumulation_steps'],
    max_seq_length=HYPERPARAMETERS['max_seq_length'],
    save_steps=HYPERPARAMETERS['save_steps'],
    eval_steps=HYPERPARAMETERS['eval_steps'],
    warmup_ratio=HYPERPARAMETERS['warmup_ratio'],
    lr_scheduler_type=HYPERPARAMETERS['lr_scheduler_type'],
    run_name=RUN_NAME,
    use_wandb=USE_WANDB,
    seed=HYPERPARAMETERS['seed'],
)
train_dataset, eval_dataset = fine_tuning.build_datasets(
    fit_train_records, fit_validation_records, tokenizer=tokenizer,
    max_train=HYPERPARAMETERS['max_train_samples'],
    max_seq_length=HYPERPARAMETERS['max_seq_length'],
    seed=HYPERPARAMETERS['seed'], test_records=test_records,
)

# Hugging Face revision for this round, keyed to how many resumes were trained.
NUM_TRAIN_RESUMES = len(train_dataset)
REVISION = fine_tuning.checkpoint_revision(NUM_TRAIN_RESUMES, run_tag=EXPERIMENT_ID)
print(train_dataset)
print(eval_dataset)
print('Trained resumes:', NUM_TRAIN_RESUMES, '| HF revision:', REVISION)
selected_records = fine_tuning.select_training_records(
    fit_train_records, HYPERPARAMETERS['max_train_samples'], seed=HYPERPARAMETERS['seed'],
)
from collections import Counter
print('Selected category counts:', dict(Counter(r.get('category') for r in selected_records)))
manifest = {
    'run_id': EXPERIMENT_ID, 'model_id': MODEL_ID, 'hyperparameters': HYPERPARAMETERS,
    'runtime_versions': runtime_versions,
    'prompt_sha256': hashlib.sha256((PROJECT_ROOT / 'src/steps/04_evaluation/evaluation_prompt.md').read_bytes()).hexdigest(),
    'schema_sha256': hashlib.sha256((PROJECT_ROOT / 'src/schema/resume_output.py').read_bytes()).hexdigest(),
    'dataset_revision': os.getenv('HF_DATASET_REVISION', 'main'),
    'split_sha256': {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in splits.items()},
    'train_resume_ids': [r['resume_id'] for r in selected_records],
    'preparation': preparation_report, 'train_lengths': train_lengths, 'validation_lengths': validation_lengths,
}
manifest_path = OUTPUT_DIR / 'run_manifest.json'
if RESUME_FROM_CHECKPOINT:
    previous = json.loads(manifest_path.read_text())
    for key in ('model_id', 'hyperparameters', 'split_sha256', 'train_resume_ids', 'runtime_versions', 'prompt_sha256', 'schema_sha256'):
        if previous[key] != manifest[key]:
            raise ValueError(f'Resume configuration changed: {key}. Start a new experiment instead.')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

## 4. Train and retain resumable checkpoints

For another experiment, rerun the configuration, fresh model-loading, dataset, and training cells. To resume, keep the original run ID and manifest settings. Checkpoints are saved every `save_steps`; interruptions can lose the progress since the latest checkpoint.

Optional baseline evaluation runs on exactly the same prepared validation records, sample seed, and output-token budget as post-training evaluation. Old charts under `evals/` may use different samples/splits and are not a direct before/after comparison.

In [ ]:
if RUN_BASELINE and not RESUME_FROM_CHECKPOINT:
    baseline_report = fine_tuning.evaluate_records(
        model, tokenizer, fit_validation_records, model_name=f'baseline:{MODEL_ID}',
        limit=EVAL_SAMPLE, max_new_tokens=MAX_NEW_TOKENS, seed=HYPERPARAMETERS['seed'],
        output_dir=OUTPUT_DIR / 'evaluation-before',
    )
    print('Baseline:', json.dumps(baseline_report.metrics, indent=2))

if USE_WANDB:
    import wandb
    if wandb.run is not None:
        wandb.finish()
    fine_tuning.configure_wandb_env(entity=WANDB_ENTITY, project=WANDB_PROJECT, run_name=RUN_NAME)
try:
    trainer = fine_tuning.train(
        model, tokenizer, train_dataset, eval_dataset,
        lora_config=lora_config, training_args=training_args,
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    )
finally:
    if USE_WANDB:
        wandb.finish()

In [ ]:
adapter_dir = fine_tuning.save_adapter(trainer, tokenizer, OUTPUT_DIR)
print('Saved adapter to:', adapter_dir)

## 5. Evaluate the trained adapter

Generation uses `trainer.model` in evaluation mode with caching enabled. The manifest, results, and metrics are saved under this experiment's output directory. If `RUN_BASELINE=True`, compare against `evaluation-before` using the same validation sample. Full semantic scoring requires `TYPESAFE_API_KEY`.

In [ ]:
report = fine_tuning.evaluate_records(
    trainer.model, tokenizer, fit_validation_records,
    model_name=f'finetuned:{MODEL_ID}', limit=EVAL_SAMPLE,
    max_new_tokens=MAX_NEW_TOKENS, seed=HYPERPARAMETERS['seed'],
    output_dir=OUTPUT_DIR / 'evaluation-after',
)
print(json.dumps(report.metrics, indent=2))
summary = report_results({f'finetuned:{MODEL_ID}': report})

## 6. Optional adapter publication

Set `PUSH_TO_HUB=True` in the reporting configuration to upload the adapter and tokenizer. Each experiment has a distinct revision containing its run ID. Checkpoint uploads are separate; the adapter export does not contain optimizer/scheduler state.

In [ ]:
if PUSH_TO_HUB:
    url = fine_tuning.push_adapter_to_hub(
        trainer.model, tokenizer, revision=REVISION,
        commit_message=f'Adapter trained on {NUM_TRAIN_RESUMES} resumes ({EXPERIMENT_ID})',
    )
    print(url)
else:
    print('Adapter saved locally; Hub publication is disabled.')

In [ ]:
# Run after evaluation/export; delete references before emptying CUDA cache.
for name in ('trainer', 'model', 'tokenizer'):
    globals().pop(name, None)
fine_tuning.free_memory()